# Week 4 — Block 2: Guided Demo (Time-Series)

**DATS 6401 · Visualization of Complex Data**

~30 min on Mauna Loa CO₂ (ships with statsmodels):

1. Index, resample, resolution (~8 min)
2. ACF → the period → the matched rolling window (~8 min)
3. Decompose; read the residual (~7 min)
4. The ±2σ band + a bootstrap interval (~7 min)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

co2 = sm.datasets.co2.load_pandas().data.dropna()
print(type(co2.index), "| span:", co2.index.min().date(), "→", co2.index.max().date())
co2.head(3)

## Part 1 — Resolution is a choice

A DatetimeIndex unlocks `resample` — and `.mean()` vs `.sum()` is itself a meaning choice (averaging CO₂ ✓; you'd SUM daily sales).

In [ ]:
monthly = co2["co2"].resample("MS").mean().dropna()
yearly  = co2["co2"].resample("YS").mean()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
co2["co2"].loc["1990":"1995"].plot(ax=axes[0], color="#2E6E8E")
axes[0].set_title("weekly slice: sawtooth dominates")
yearly.plot(ax=axes[1], color="#2E6E8E")
axes[1].set_title("yearly: only the trend survives")
plt.show()

## Part 2 — Find the period, then smooth with it

Don't guess the window: **ask the ACF.**

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, ax = plt.subplots(figsize=(8, 3))
plot_acf(monthly.diff().dropna(), lags=36, ax=ax)
ax.set_title("ACF of monthly CHANGES: spikes at 12, 24, 36 → period = 12")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
seg = monthly.loc["1980":"1995"]
seg.plot(ax=ax, alpha=0.3, color="#5a6672", label="monthly")
seg.rolling(12).mean().plot(ax=ax, lw=2, color="#2E6E8E", label="w=12 (one full cycle)")
ax.legend(); ax.set_title("The matched window: seasonality cancels OUT of the trend")
plt.show()

**Live experiment:** change 12 → 3, rerun (noise survives); → 60 (the trend's curvature dies). The window is now a *decision*, not a default.

## Part 3 — Decomposition

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

res = seasonal_decompose(monthly, period=12)
fig = res.plot(); fig.set_size_inches(8.5, 5.5)
plt.tight_layout(); plt.show()

**Read it bottom-up with the class:** residual ≈ structureless noise → the additive model fit. Then the question: *what would a multiplicative series' residual look like under this additive model?* (Growing wedge.)

## Part 4 — Uncertainty, two ways

In [ ]:
roll = monthly.rolling(12)
mean, std = roll.mean(), roll.std()
fig, ax = plt.subplots(figsize=(9, 3.2))
mean.plot(ax=ax, color="#2E6E8E", label="12-mo mean")
ax.fill_between(mean.index, mean - 2*std, mean + 2*std, alpha=0.2, color="#2E6E8E", label="±2σ")
ax.legend(); ax.set_title("Band #1: data spread around the smooth line")
plt.show()

In [ ]:
# Band #2 — bootstrap CI for a STATISTIC (mean monthly increase in the 1990s)
rng = np.random.default_rng(0)
changes = monthly.diff().dropna().loc["1990":"1999"].values
boots = [rng.choice(changes, len(changes), replace=True).mean() for _ in range(2000)]
lo, hi = np.percentile(boots, [2.5, 97.5])
fig, ax = plt.subplots(figsize=(8, 2.6))
ax.hist(boots, bins=40, color="#2E6E8E", alpha=0.85)
for v in (lo, hi): ax.axvline(v, color="#d9534f", lw=2)
ax.set_title(f"Bootstrap: mean 1990s monthly rise, 95% CI [{lo:.3f}, {hi:.3f}] ppm")
plt.show()

**Narrate the distinction once more:** ±2σ answers "how far do values stray"; the CI answers "how sure are we about the mean". Different questions, both called 'a band' — captions must say which.

## Wrap-up → Block 3

Pipeline: **resample → ACF → matched window → decompose → band.** Your series next.